16c. Tổng hợp GĐ G: ablation hybrid, Pareto, ablation GAT, tinh chỉnh, bản cải tiến, ngưỡng, hình, CPU (task 5.2, 5.3, 5.4, 5.5, 5.8)

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

ALLOW_GPU = False
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    """Đường dẫn đầu tiên khớp glob đệ quy dưới /kaggle/input (báo lỗi rõ nếu không có)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"])
assert ALLOW_GPU or INFO["device"] == "CPU", "Chưa được phép dùng GPU: tắt GPU trong cài đặt notebook"

## Đường dẫn đầu vào (output của 16a, 16d, 16e, 16g, 16, 16f) và kết quả cũ trong repo (GĐ D, E, F)

In [ ]:
def where(name, required=True):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits and required:
        raise AssertionError(f"Không thấy {name}")
    return os.path.dirname(hits[0]) if hits else None


SRC = {"bot": where("pareto_raw_nf_bot_iot_v3.csv"), "ton_hybrid": where("pareto_raw_nf_ton_iot_v3.csv"),
       "ton_thr": where("thresholds_base_gnn_nf_ton_iot_v3.json"), "ton_tuning": where("grid_nf_ton_iot_v3.csv", False)}
grids = sorted(glob.glob("/kaggle/input/**/grid_nf_ton_iot_v3.csv", recursive=True))
print("grid_nf_ton_iot_v3.csv:", grids)
# notebook 16 (arch + improved) và 16g (tuning GCN, GraphSAGE) cùng ghi grid_nf_ton_iot_v3.csv: phân biệt theo nội dung
import pandas as pd
ton_grid = ton_tuning = None
for p in grids:
    parts = set(pd.read_csv(p)["part"])
    if "arch" in parts:
        ton_grid = os.path.dirname(p)
    elif "tuning" in parts:
        ton_tuning = os.path.dirname(p)
SRC["ton_grid"], SRC["ton_tuning"] = ton_grid, ton_tuning
SRC["windows"] = where("grid_nf_ton_iot_v3_w1.csv", False)
INFO["sources"] = SRC
save_info()
print(json.dumps(SRC, indent=1))

## Tổng hợp: bảng, thresholds.json, hình

In [ ]:
args = [f"{k}={v}" for k, v in SRC.items() if v]
with step("aggregate_g"):
    run([sys.executable, "-I", "scripts/aggregate_g.py", WORK, f"repo={REPO}", *args], cwd=REPO)

In [ ]:
print(sorted(os.listdir(WORK)))
INFO["outputs"] = sorted(os.listdir(WORK))
INFO["summary"] = json.load(open(f"{WORK}/aggregate_summary.json", encoding="utf-8"))
save_info()
print(open(f"{WORK}/run_info.json", encoding="utf-8").read()[:6000])